# Assignment 2 - Final Focused Model Selection

This notebook narrows expanded validation and 2025 holdout testing to the retained final model set. It intentionally does not rerun every exploratory model from the research notebooks.

## Selection Insight

The prototype results should be used only to choose promising model families. They should not be treated as final model evidence. The retained set tests the strongest sanity baselines, direct nonlinear cluster-aware models, the scalable LightGBM candidate, the RandomForest robustness check, and the hierarchical zone-allocation strategy.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

repo_root = Path.cwd()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / "data").exists() and (candidate / "solution" / "assignment_2" / "src").exists():
        repo_root = candidate
        break
else:
    raise FileNotFoundError("Could not find ECESIS repository root from notebook path.")

src_dir = repo_root / "solution" / "assignment_2" / "src"
outputs_dir = repo_root / "solution" / "assignment_2" / "outputs"
sys.path.insert(0, str(src_dir))
pd.set_option("display.max_columns", None)


## Run Mode

The final-selection artifacts have already been generated on a deterministic focused bus subset. The notebook reads them by default. Set `RUN_FINAL_SELECTION = True` to rerun the focused validation and 2025 holdout workflow.

In [ ]:
RUN_FINAL_SELECTION = False

if RUN_FINAL_SELECTION:
    from final_model_selection import run_final_model_selection
    final_outputs = run_final_model_selection(outputs_dir, n_buses=6)
    result_shapes = {k: v.shape for k, v in final_outputs.items()}
else:
    result_shapes = {
        name: pd.read_csv(outputs_dir / name).shape
        for name in [
            "final_model_validation_results.csv",
            "final_2025_holdout_results.csv",
            "final_evaluation_scope.csv",
            "final_per_zone_wmape.csv",
            "final_per_cluster_wmape.csv",
            "final_per_bus_error_distribution.csv",
        ]
    }
result_shapes

## Retained and Deferred Models

In [ ]:
retained = pd.read_csv(outputs_dir / "final_retained_model_rationale.csv")
deferred = pd.read_csv(outputs_dir / "final_deferred_model_rationale.csv")
retained, deferred

## Expanded Validation and 2025 Holdout

In [ ]:
validation = pd.read_csv(outputs_dir / "final_model_validation_results.csv")
holdout = pd.read_csv(outputs_dir / "final_2025_holdout_results.csv")
scope = pd.read_csv(outputs_dir / "final_evaluation_scope.csv")
scope, validation.sort_values(["fold_name", "horizon", "level", "wmape"]).head(30), holdout.sort_values(["horizon", "level", "wmape"]).head(30)

## Decision Criteria

The recommendation should be based on 2025 holdout WMAPE, stability across 2023/2024/2025, ability to beat `baseline_lag_168h`, runtime/scalability, and operational defensibility. On the focused subset, the weekly lag baseline remains strongest on the 2025 next-day holdout, so no ML model should be declared final yet.

In [ ]:
summary_path = outputs_dir / "final_model_selection_summary.md"
print(summary_path.read_text()[:3000])

## Diagnostics

In [ ]:
per_zone = pd.read_csv(outputs_dir / "final_per_zone_wmape.csv")
per_cluster = pd.read_csv(outputs_dir / "final_per_cluster_wmape.csv")
per_bus = pd.read_csv(outputs_dir / "final_per_bus_error_distribution.csv")
per_zone.head(), per_cluster.head(), per_bus.head()